# Detection / YOLO GC10-DET Evidence Notebook

## Purpose
This notebook presents governed Detection / YOLO evidence for the GC10-DET object detection track. It is Colab-first, local-capable, fallback-aware, reproducibility-aware, leakage-aware, review-friendly, output-aware, and aligned with the governed project structure.

## Scope
- Track: detection
- Task: object detection
- Dataset: GC10-DET governed detection dataset
- Run: `yolo_train_v0_1_0`
- Evaluation split: validation

## What This Notebook Does
- Loads existing governed Detection/YOLO artifacts.
- Presents metrics, training curves, YOLO visual outputs, registry linkage, re-audit status, and frontend-ready evidence references.
- Clearly separates governance/evidence status from model-readiness status.

## What This Notebook Does Not Do
- It does not train YOLO, evaluate YOLO, generate artifacts, update registries, run builders, or rewrite timestamps.
- It does not replace source code, configs, validation scripts, registries, or governed artifacts.
- It does not invent metrics, hide limitations, or claim production readiness.

This notebook is not the source of truth. It consumes governed artifacts only.

## 1. Runtime Detection
Detect local vs Colab execution, Python version, repository root, and safe runtime status. This section does not query accelerators for training and does not write files.

In [ ]:
from pathlib import Path
import csv
import json
import platform
import sys

try:
    import yaml
except Exception:
    yaml = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

IS_COLAB = 'google.colab' in sys.modules
print(f'runtime={"colab" if IS_COLAB else "local"}')
print(f'python={platform.python_version()}')
print(f'platform={platform.platform()}')
print('device_summary=not queried; notebook is read-only evidence presentation')

## 2. Path And Artifact Resolution
In Colab, clone or mount the repository first, then run from the repository root or update `REPO_ROOT` manually. Required missing evidence fails clearly; optional YOLO visuals are reported as unavailable.

In [ ]:
PROJECT_MARKERS = ['artifacts', 'scripts', 'notebooks']

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in PROJECT_MARKERS):
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd()).resolve()
RUN_ID = 'yolo_train_v0_1_0'
TRACK_ID = 'detection'
TASK_TYPE = 'object_detection'
DATASET_ID = 'gc10det_detection'
DATASET_VERSION = 'gc10det_1.0'
MODEL_NAME = 'yolo'

def rel(path: str) -> Path:
    return REPO_ROOT / path

ARTIFACTS = [
    {'key': 'yolo_run_dir', 'required': True, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0', 'kind': 'directory'},
    {'key': 'best_checkpoint', 'required': True, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/weights/best.pt', 'kind': 'checkpoint'},
    {'key': 'last_checkpoint', 'required': True, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/weights/last.pt', 'kind': 'checkpoint'},
    {'key': 'results_csv', 'required': True, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/results.csv', 'kind': 'csv'},
    {'key': 'args_yaml', 'required': True, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/args.yaml', 'kind': 'yaml'},
    {'key': 'training_result', 'required': True, 'path': 'artifacts/models/analysis/training_result__yolo_train_v0_1_0.json', 'kind': 'json'},
    {'key': 'evaluation_summary', 'required': True, 'path': 'artifacts/models/metrics/detection_evaluation__yolo_train_v0_1_0__validation.json', 'kind': 'json'},
    {'key': 'artifact_inventory', 'required': True, 'path': 'artifacts/models/inventory/track_detection_artifact_inventory__yolo_train_v0_1_0.json', 'kind': 'json'},
    {'key': 'metadata_summary', 'required': True, 'path': 'artifacts/models/metadata/track_detection_yolo_metadata_summary__yolo_train_v0_1_0.json', 'kind': 'json'},
    {'key': 'posthoc_log', 'required': True, 'path': 'artifacts/models/logs/track_detection_yolo_posthoc_run_log__yolo_train_v0_1_0.json', 'kind': 'json'},
    {'key': 'run_registry', 'required': True, 'path': 'artifacts/models/registry/run_registry.yaml', 'kind': 'yaml'},
    {'key': 'artifact_registry', 'required': True, 'path': 'artifacts/models/registry/artifact_registry.yaml', 'kind': 'yaml'},
    {'key': 'reaudit_report', 'required': True, 'path': 'artifacts/reports/audits/detection_yolo_reaudit__yolo_train_v0_1_0.json', 'kind': 'json'},
    {'key': 'results_png', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/results.png', 'kind': 'image'},
    {'key': 'confusion_matrix_png', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/confusion_matrix.png', 'kind': 'image'},
    {'key': 'confusion_matrix_normalized_png', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/confusion_matrix_normalized.png', 'kind': 'image'},
    {'key': 'box_f1_curve_png', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/BoxF1_curve.png', 'kind': 'image'},
    {'key': 'box_pr_curve_png', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/BoxPR_curve.png', 'kind': 'image'},
    {'key': 'box_p_curve_png', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/BoxP_curve.png', 'kind': 'image'},
    {'key': 'box_r_curve_png', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/BoxR_curve.png', 'kind': 'image'},
    {'key': 'val_batch0_labels', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/val_batch0_labels.jpg', 'kind': 'image'},
    {'key': 'val_batch0_pred', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/val_batch0_pred.jpg', 'kind': 'image'},
    {'key': 'val_batch1_labels', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/val_batch1_labels.jpg', 'kind': 'image'},
    {'key': 'val_batch1_pred', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/val_batch1_pred.jpg', 'kind': 'image'},
    {'key': 'val_batch2_labels', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/val_batch2_labels.jpg', 'kind': 'image'},
    {'key': 'val_batch2_pred', 'required': False, 'path': 'artifacts/detection/yolo/runs/yolo_train_v0_1_0/val_batch2_pred.jpg', 'kind': 'image'},
]
ARTIFACT_BY_KEY = {item['key']: item for item in ARTIFACTS}

print(f'repo_root={REPO_ROOT}')
for item in ARTIFACTS:
    path = rel(item['path'])
    print(f"{item['key']}: exists={path.exists()} required={item['required']} path={item['path']}")

## 3. Config, Dataset, And Track Summary
This section states the governed Detection / YOLO identity used by the notebook. Canonical evidence is loaded from governed JSON/YAML files in later sections.

In [ ]:
TRACK_SUMMARY = {
    'track_id': TRACK_ID,
    'task_type': TASK_TYPE,
    'dataset_id': DATASET_ID,
    'dataset_version': DATASET_VERSION,
    'model_name': MODEL_NAME,
    'run_id': RUN_ID,
    'evaluation_split': 'validation',
    'model_readiness': 'not_ready',
}
if pd:
    display(pd.DataFrame([TRACK_SUMMARY]))
else:
    print(json.dumps(TRACK_SUMMARY, indent=2))

## 4. Artifact Loading Helpers
Concise helpers load JSON, YAML, and CSV evidence. These helpers do not implement training, evaluation, registry updates, or re-audit generation.

In [ ]:
def artifact_path(key: str) -> Path:
    return rel(ARTIFACT_BY_KEY[key]['path'])

def load_json_artifact(key: str, required: bool = True):
    path = artifact_path(key)
    if not path.exists():
        if required:
            raise FileNotFoundError(f'missing required JSON artifact: {key} -> {ARTIFACT_BY_KEY[key]["path"]}')
        print(f'missing optional JSON artifact: {key} -> {ARTIFACT_BY_KEY[key]["path"]}')
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def load_yaml_artifact(key: str, required: bool = True):
    path = artifact_path(key)
    if not path.exists():
        if required:
            raise FileNotFoundError(f'missing required YAML artifact: {key} -> {ARTIFACT_BY_KEY[key]["path"]}')
        print(f'missing optional YAML artifact: {key} -> {ARTIFACT_BY_KEY[key]["path"]}')
        return None
    if yaml is None:
        raise RuntimeError('PyYAML is required to read registry/args YAML in this notebook environment')
    return yaml.safe_load(path.read_text(encoding='utf-8'))

def load_csv_rows(key: str, required: bool = True):
    path = artifact_path(key)
    if not path.exists():
        if required:
            raise FileNotFoundError(f'missing required CSV artifact: {key} -> {ARTIFACT_BY_KEY[key]["path"]}')
        print(f'missing optional CSV artifact: {key} -> {ARTIFACT_BY_KEY[key]["path"]}')
        return []
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

def assert_required_artifacts_present():
    missing = [item for item in ARTIFACTS if item['required'] and not rel(item['path']).exists()]
    if missing:
        details = '\n'.join(f"- {item['key']}: {item['path']}" for item in missing)
        raise FileNotFoundError(f'Missing required Detection/YOLO evidence artifacts:\n{details}')
    print('required_artifacts_status=pass')

def show_table(rows):
    if pd:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(json.dumps(row, indent=2))

def nested_get(data, path, default=None):
    current = data
    for part in path:
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current

assert_required_artifacts_present()

## 5. Detection Evidence Inventory
Required governed evidence must exist. Optional YOLO visual outputs are listed honestly and may be unavailable depending on the run output.

In [ ]:
inventory_rows = []
for item in ARTIFACTS:
    path = rel(item['path'])
    inventory_rows.append({
        'artifact_key': item['key'],
        'required': item['required'],
        'exists': path.exists(),
        'kind': item['kind'],
        'size_bytes': path.stat().st_size if path.exists() and path.is_file() else None,
        'path': item['path'],
    })
show_table(inventory_rows)

## 6. Training Result Summary
Load the governed training result summary. This was a first governed 1-epoch baseline run and must not be presented as production-ready.

In [ ]:
training_result = load_json_artifact('training_result')
training_row = {
    'run_id': training_result.get('run_id'),
    'track_id': training_result.get('track_id'),
    'task_type': training_result.get('task_type'),
    'model_name': nested_get(training_result, ['model', 'model_name']),
    'model_version': nested_get(training_result, ['model', 'model_version']),
    'dataset_id': nested_get(training_result, ['dataset', 'dataset_id']),
    'dataset_version': nested_get(training_result, ['dataset', 'dataset_version']),
    'config_id': nested_get(training_result, ['config', 'config_id']),
    'training_status': training_result.get('training_status'),
    'execution_environment': training_result.get('execution_environment'),
    'epochs': nested_get(training_result, ['training_parameters', 'epochs']) or nested_get(training_result, ['planned_config_parameters', 'epochs']),
    'mAP50': nested_get(training_result, ['metrics', 'mAP50']),
    'mAP50_95': nested_get(training_result, ['metrics', 'mAP50_95']),
    'inventory_status': nested_get(training_result, ['artifacts', 'inventory_status']),
    'registry_updated': nested_get(training_result, ['governance', 'registry_updated']),
}
show_table([training_row])
print('baseline_note=first governed 1-epoch YOLO baseline; not production-ready evidence')

## 7. Detection Validation Metrics
Load the governed detection validation evaluation. Governance/evidence can pass while model-readiness remains not ready.

In [ ]:
evaluation = load_json_artifact('evaluation_summary')
eval_metrics = evaluation.get('metrics', {})
metric_interpretation = evaluation.get('metric_interpretation', {})
metric_row = {
    'precision': eval_metrics.get('precision'),
    'recall': eval_metrics.get('recall'),
    'mAP50': eval_metrics.get('mAP50'),
    'mAP50_95': eval_metrics.get('mAP50_95'),
    'performance_level': metric_interpretation.get('performance_level'),
    'production_readiness': metric_interpretation.get('production_readiness'),
    'evaluation_status': evaluation.get('evaluation_status'),
    'evaluation_split': evaluation.get('evaluation_split'),
    'model_readiness': 'not_ready',
}
show_table([metric_row])
print('model_readiness=not_ready')
print('production_ready=false')

## 8. YOLO Training Curves / Results CSV
Load `results.csv`, report schema and row count, and plot available loss/metric columns with matplotlib when possible. No CSV is generated or rewritten.

In [ ]:
results_rows = load_csv_rows('results_csv')
columns = list(results_rows[0].keys()) if results_rows else []
print(f'results_csv_rows={len(results_rows)}')
print(f'results_csv_columns={columns}')
show_table(results_rows[:5] if results_rows else [{'status': 'results.csv empty or unavailable'}])

def to_float(value):
    try:
        return float(value)
    except Exception:
        return None

plot_columns = [col for col in columns if col != 'epoch']
epochs = [to_float(row.get('epoch')) for row in results_rows]
if plt and results_rows and epochs:
    selected = [col for col in plot_columns if any(token in col for token in ['loss', 'mAP', 'precision', 'recall'])]
    for group_name, group_cols in {
        'YOLO Loss Curves': [col for col in selected if 'loss' in col],
        'YOLO Metric Curves': [col for col in selected if 'loss' not in col],
    }.items():
        if not group_cols:
            continue
        plt.figure(figsize=(8, 4))
        for col in group_cols:
            values = [to_float(row.get(col)) for row in results_rows]
            if any(value is not None for value in values):
                plt.plot(epochs, values, marker='o', label=col)
        plt.title(group_name)
        plt.xlabel('Epoch')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print('results_csv_plot_status=matplotlib_unavailable_or_no_rows')

## 9. YOLO Visual Evidence
List optional YOLO visual outputs and display a small number of existing images when matplotlib can safely read them. Missing optional images are not treated as success.

In [ ]:
visual_keys = [item['key'] for item in ARTIFACTS if item['kind'] == 'image']
visual_rows = []
for key in visual_keys:
    path = artifact_path(key)
    visual_rows.append({'artifact_key': key, 'exists': path.exists(), 'size_bytes': path.stat().st_size if path.exists() else None, 'path': ARTIFACT_BY_KEY[key]['path']})
show_table(visual_rows)

if plt:
    try:
        import matplotlib.image as mpimg
        display_keys = [key for key in ['results_png', 'confusion_matrix_png', 'confusion_matrix_normalized_png', 'box_pr_curve_png', 'val_batch0_labels', 'val_batch0_pred'] if artifact_path(key).exists()]
        for key in display_keys[:6]:
            path = artifact_path(key)
            plt.figure(figsize=(8, 5))
            plt.imshow(mpimg.imread(path))
            plt.title(key)
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        print(f'yolo_visuals_displayed={len(display_keys[:6])}')
    except Exception as exc:
        print(f'yolo_visual_display_status=unavailable: {exc}')
else:
    print('yolo_visual_display_status=matplotlib_unavailable')

## 10. Registry And Governance Linkage
Read the run and artifact registries without modifying them. The notebook checks the run count and expected Detection artifact IDs for presentation only.

In [ ]:
run_registry = load_yaml_artifact('run_registry')
artifact_registry = load_yaml_artifact('artifact_registry')
expected_artifact_ids = [
    'track_detection__yolo_train_v0_1_0__training_result',
    'track_detection__yolo_train_v0_1_0__validation_evaluation',
    'track_detection__yolo_train_v0_1_0__artifact_inventory',
    'track_detection__yolo_train_v0_1_0__metadata_summary',
    'track_detection__yolo_train_v0_1_0__posthoc_log',
    'track_detection__yolo_train_v0_1_0__best_checkpoint',
    'track_detection__yolo_train_v0_1_0__last_checkpoint',
    'track_detection__yolo_train_v0_1_0__training_metrics_csv',
    'track_detection__yolo_train_v0_1_0__training_args',
]
runs = run_registry.get('runs', []) if isinstance(run_registry, dict) else []
run_matches = [run for run in runs if run.get('run_id') == RUN_ID]
artifacts = artifact_registry.get('artifacts', []) if isinstance(artifact_registry, dict) else []
artifact_rows = []
for artifact_id in expected_artifact_ids:
    matches = [item for item in artifacts if item.get('artifact_id') == artifact_id]
    artifact_rows.append({'artifact_id': artifact_id, 'matches': len(matches), 'present_once': len(matches) == 1})
show_table([{'run_id': RUN_ID, 'run_registry_matches': len(run_matches), 'present_once': len(run_matches) == 1}])
show_table(artifact_rows)
print('registry_write_status=not_modified_by_notebook')

## 11. Final Re-Audit Report Summary
Load the committed final re-audit report without regenerating it. The report separates governance pass from model readiness.

In [ ]:
reaudit = load_json_artifact('reaudit_report')
reaudit_row = {
    'audit_status': reaudit.get('audit_status'),
    'validation_script_status': nested_get(reaudit, ['governance_status', 'validation_script_status']),
    'governance_pipeline_pass': nested_get(reaudit, ['decision', 'governance_pipeline_pass']),
    'model_production_ready': nested_get(reaudit, ['decision', 'model_production_ready']),
    'detection_track_final_pass': nested_get(reaudit, ['decision', 'detection_track_final_pass']),
    'production_readiness': nested_get(reaudit, ['model_performance_status', 'production_readiness']),
    'performance_level': nested_get(reaudit, ['model_performance_status', 'performance_level']),
    'mAP50': nested_get(reaudit, ['model_performance_status', 'mAP50']),
    'mAP50_95': nested_get(reaudit, ['model_performance_status', 'mAP50_95']),
}
show_table([reaudit_row])
print('governance_evidence_status=pass')
print('model_readiness_status=not_ready')

## 12. Frontend-Ready Artifact Summary
Frontend applications should consume structured JSON artifacts and image files directly, not notebook cells or logs.

In [ ]:
frontend_keys = ['evaluation_summary', 'training_result', 'artifact_inventory', 'metadata_summary', 'posthoc_log', 'reaudit_report']
frontend_rows = []
for key in frontend_keys:
    item = ARTIFACT_BY_KEY[key]
    frontend_rows.append({'artifact_key': key, 'frontend_use': 'structured JSON evidence', 'exists': artifact_path(key).exists(), 'path': item['path']})
for key in visual_keys:
    item = ARTIFACT_BY_KEY[key]
    frontend_rows.append({'artifact_key': key, 'frontend_use': 'optional visual asset', 'exists': artifact_path(key).exists(), 'path': item['path']})
show_table(frontend_rows)

## 13. CI/CD And MLOps Evidence Summary
Summarize traceability from config to run, checkpoints/artifacts, metrics, metadata, registry, validator, and re-audit. Generator scripts are referenced but not executed.

In [ ]:
metadata_summary = load_json_artifact('metadata_summary')
posthoc_log = load_json_artifact('posthoc_log')
artifact_inventory = load_json_artifact('artifact_inventory')
mlops_row = {
    'run_id': RUN_ID,
    'dataset_id': DATASET_ID,
    'dataset_version': DATASET_VERSION,
    'config_id': 'yolo_train_v0_1_0',
    'model_name': MODEL_NAME,
    'run_status': metadata_summary.get('run_status') or posthoc_log.get('run_status'),
    'metrics_path': ARTIFACT_BY_KEY['evaluation_summary']['path'],
    'best_checkpoint_path': ARTIFACT_BY_KEY['best_checkpoint']['path'],
    'last_checkpoint_path': ARTIFACT_BY_KEY['last_checkpoint']['path'],
    'artifact_inventory_path': ARTIFACT_BY_KEY['artifact_inventory']['path'],
    'metadata_path': ARTIFACT_BY_KEY['metadata_summary']['path'],
    'run_registry_path': ARTIFACT_BY_KEY['run_registry']['path'],
    'artifact_registry_path': ARTIFACT_BY_KEY['artifact_registry']['path'],
    'validator_script': 'scripts/validation/validate_detection_artifacts.py',
    'reaudit_report': ARTIFACT_BY_KEY['reaudit_report']['path'],
    'reaudit_builder_reference_not_run': 'scripts/validation/build_detection_reaudit_report.py',
}
show_table([mlops_row])
print('traceability_chain=config -> run -> checkpoint/artifacts -> metrics -> metadata -> registry -> validator/re-audit')
print('generator_execution_status=not_run_by_notebook')

## 14. Limitations
- This notebook is an evidence/presentation layer, not canonical training or evaluation logic.
- No YOLO training, evaluation generation, registry update, re-audit builder, or timestamp rewrite is executed here.
- Outputs are based on existing governed artifacts only.
- This is a first governed 1-epoch YOLO baseline run.
- Current metrics are weak and not production-ready: precision `0.00477`, recall `0.54003`, mAP50 `0.04518`, mAP50_95 `0.01651`.
- Detection governance/evidence PASS is not the same as model-readiness PASS.
- `model_production_ready = false`.
- `detection_track_final_pass = false`.

## 15. Final Decision Summary
Report Detection governance and model-readiness status from existing governed evidence. This notebook does not create a new production decision.

In [ ]:
final_row = {
    'detection_governance_evidence_status': 'pass' if nested_get(reaudit, ['decision', 'governance_pipeline_pass']) is True else 'review_required',
    'detection_model_readiness_status': nested_get(reaudit, ['model_performance_status', 'production_readiness']) or 'not_ready',
    'precision': eval_metrics.get('precision'),
    'recall': eval_metrics.get('recall'),
    'mAP50': eval_metrics.get('mAP50'),
    'mAP50_95': eval_metrics.get('mAP50_95'),
    'production_ready': nested_get(reaudit, ['decision', 'model_production_ready']),
    'detection_track_final_pass': nested_get(reaudit, ['decision', 'detection_track_final_pass']),
    'remaining_required_work': 'stronger YOLO training run; stronger validation/test evaluation; re-audit before any production-ready claim',
    'next_step_after_notebook_review': 'update roadmap/status or proceed to script cloud/local readiness audit',
}
show_table([final_row])